In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
data = pd.read_csv("../data/processed/kc_house_features.csv")
data["zipcode"] = data["zipcode"].astype(str)

In [3]:
FEATURES = [
    "sqft_living",
    "sqft_lot",
    "bedrooms",
    "bathrooms",
    "floors",
    "waterfront",
    "view",
    "condition",
    "grade",
    "age",
    "renovated",
    "zipcode",
]
X = data[FEATURES]
y = data["log_price"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape[0], "Test:", X_test.shape[0])

Train: 17273 Test: 4319


In [5]:
def make_preprocessor():
    return ColumnTransformer(
        [("zip", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["zipcode"])],
        remainder="passthrough",
    )

In [6]:
models = {
    "Linear regression": LinearRegression(),
    "Gradient boosting": HistGradientBoostingRegressor(random_state=42),
}

In [7]:
results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", model)])
    pipe.fit(X_train, y_train)
    pred_dollars = np.exp(pipe.predict(X_test))
    real_dollars = np.exp(y_test)
    mae = mean_absolute_error(real_dollars, pred_dollars)
    r2 = r2_score(y_test, pipe.predict(X_test))
    results[name] = {"pipe": pipe, "mae": mae}
    print(name, "- MAE:", round(mae), "dollars - R2:", round(r2, 3))

Linear regression - MAE: 77604 dollars - R2: 0.873
Gradient boosting - MAE: 74473 dollars - R2: 0.878


In [8]:
best_name = min(results, key=lambda n: results[n]["mae"])
print("\nBest model:", best_name)


Best model: Gradient boosting


In [9]:
joblib.dump(
    {"pipeline": results[best_name]["pipe"], "features": FEATURES},
    "../models/house_model.joblib",
)

['../models/house_model.joblib']